In [2]:
import matplotlib.pyplot as plt
import sys
sys.path.append('../../Codes/library/')

from funcs import*

plt.rcParams['text.usetex'] = True
Text_files_path = '/Users/robertomorantovar/Dropbox/Research/Immune_system/'
output_plot = '/Users/robertomorantovar/Dropbox/My_Documents/Science/Projects/Immune_System/_Repository/Figures/exponential_proofreading/antigen_presentation/'
os.makedirs(output_plot, exist_ok=True)

%autosave 60

Autosaving every 60 seconds


In [3]:
def simulate_exact(alpha, lam, delta, x0, T=None, Theta=None, rng=np.random):
    t = 0.0
    n = x0
    traj = [(t, n)]
    while True:
        if T is not None and t >= T: break
        if Theta is not None and n >= Theta: break

        # birth clock (NHPP)
        Eb = rng.exponential(1.0)
        tau_b = (1.0/lam) * np.log(1.0 + (lam/alpha)*Eb*np.exp(-lam*t))

        # death clock
        if n > 0:
            tau_d = rng.exponential(1.0/(delta*n))
        else:
            tau_d = np.inf

        if tau_b < tau_d:
            t += tau_b
            n += 1
        else:
            t += tau_d
            n -= 1

        traj.append((t, n))

    return traj  # also return division time if Theta used

In [4]:
def simulate_tauleap(alpha, lam, delta, x0, T=None, Theta=None,
                     eps=0.05, eta=0.2, dt_max=np.inf, rng=np.random):
    t = 0.0
    n = x0
    traj = [(t, n)]
    while True:
        if T is not None and t >= T: break
        if Theta is not None and n >= Theta: break

        dt1 = (1.0/lam) * np.log(1.0 + eps)         # control exp-growth change
        dt2 = eta / delta                            # control death fraction
        dt  = min(dt1, dt2, dt_max)

        # don't step past the end time
        if T is not None:
            dt = min(dt, T - t)
            if dt <= 0: break

        # births: Poisson with integrated intensity
        m_b = (alpha/lam) * (np.exp(lam*(t+dt)) - np.exp(lam*t))
        B = rng.poisson(m_b)

        # deaths: Binomial (nonnegative guarantee)
        p = 1.0 - np.exp(-delta*dt)
        D = rng.binomial(n, p) if n > 0 else 0

        n = n + B - D
        t = t + dt
        traj.append((t, n))

    return traj

In [ ]:
fig, ax = plt.subplots(figsize=(8*1.62,8), gridspec_kw={'left':0.12, 'right':.95, 'bottom':.15, 'top': 0.94})
for i in range(10):
    traj1 = simulate_exact(alpha=1e-6, lam=1, delta=0.01, x0=1, T=100)
    traj2 = simulate_tauleap(alpha=1e-6, lam=1, delta=0.01, x0=1, T=100)
    ax.step(*zip(*traj1), color = my_blue)
    ax.step(*zip(*traj2), color = my_green)

ax.set_yscale('log')
ax.legend()